# Phase 2 — Typage des données et analyse des anomalies

Objectifs :
- Recharger les lignes CSV valides identifiées en phase 1.
- Convertir les durées et coordonnées en nombres.
- Convertir les dates au format datetime.
- Ne supprimer aucune ligne.
- Identifier, compter et afficher les valeurs qui ne peuvent pas être converties.
- Rechercher des valeurs convertibles mais incohérentes.

## I. Imports, chemin et noms de colonnes

In [2]:
from pathlib import Path
import csv
import pandas as pd

DATA_PATH = Path("../data/releves_klaxo3.csv")

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## II. Recharger le CSV de manière robuste

In [6]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, lignes in enumerate(reader, start=1):
        if len(lignes) == len(COLUMNS):
            lignes_valides.append(lignes)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(lignes),
                "contenu": lignes
            })

df_raw = pd.DataFrame(lignes_valides, columns=COLUMNS)
df_problemes_chargement = pd.DataFrame(lignes_problemes)

print(f"Lignes chargées dans le DataFrame : {len(df_raw)}")
print(f"Lignes isolées à cause de leur structure : {len(df_problemes_chargement)}")

Lignes chargées dans le DataFrame : 88679
Lignes isolées à cause de leur structure : 196


## III. Vérification de l’état initial

In [7]:
print("Dimensions :", df_raw.shape)
print("\nColonnes :")
print(df_raw.columns.tolist())

print("\nTypes avant conversion :")
print(df_raw.dtypes)

df_raw.head()

Dimensions : (88679, 11)

Colonnes :
['datetime', 'city', 'state', 'country', 'shape', 'duration_seconds', 'duration_hours_min', 'comments', 'date_posted', 'latitude', 'longitude']

Types avant conversion :
datetime              object
city                  object
state                 object
country               object
shape                 object
duration_seconds      object
duration_hours_min    object
comments              object
date_posted           object
latitude              object
longitude             object
dtype: object


,datetime,city,state,country,shape,duration_seconds,duration_hours_min,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.9411111
1,10/10/1949 21:00,lackland afb,tx,,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.6458333
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.8036111


## IV. Conservation d'une copie avant conversion

In [8]:
df = df_raw.copy()
df_avant_conversion = df.copy()

## V. Définition des colonnes à convertir

In [9]:
colonnes_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
]

colonnes_dates = [
    "datetime",
    "date_posted",
]

## VI. Convertir les nombres

In [10]:
for col in colonnes_numeriques:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## VII. Convertir les dates

In [11]:
for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], errors="coerce")

## VIII. Vérifier les types après conversion

In [12]:
df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_seconds             float64
duration_hours_min            object
comments                      object
date_posted           datetime64[ns]
latitude                     float64
longitude                    float64
dtype: object

## IX. Compter les échecs de conversion

In [13]:
resume_conversion = []

for col in colonnes_numeriques + colonnes_dates:
    valeurs_originales = df_avant_conversion[col].astype("string").str.strip()

    valeur_etait_presente = (
        valeurs_originales.notna()
        & valeurs_originales.ne("")
    )

    echec_conversion = valeur_etait_presente & df[col].isna()

    resume_conversion.append({
        "colonne": col,
        "type_final": str(df[col].dtype),
        "valeurs_presentes_avant_conversion": int(valeur_etait_presente.sum()),
        "echecs_conversion": int(echec_conversion.sum()),
        "exemples_de_valeurs_fautives": valeurs_originales[echec_conversion]
            .drop_duplicates()
            .head(10)
            .tolist()
    })

df_resume_conversion = pd.DataFrame(resume_conversion)
df_resume_conversion

,colonne,type_final,valeurs_presentes_avant_conversion,echecs_conversion,exemples_de_valeurs_fautives
0,duration_seconds,float64,88677,3,"[2`, 8`, 0.5`]"
1,latitude,float64,88679,1,[33q.200088]
2,longitude,float64,88679,0,[]
3,datetime,datetime64[ns],88679,1220,"[10/10/2005 24:00, 10/11/1994 24:00, 10/11/200..."
4,date_posted,datetime64[ns],88679,0,[]


## X. Créer le détail complet des valeurs fautives

In [14]:
anomalies_conversion = []

for col in colonnes_numeriques + colonnes_dates:
    valeurs_originales = df_avant_conversion[col].astype("string").str.strip()

    valeur_etait_presente = (
        valeurs_originales.notna()
        & valeurs_originales.ne("")
    )

    echec_conversion = valeur_etait_presente & df[col].isna()

    for index, valeur in valeurs_originales[echec_conversion].items():
        anomalies_conversion.append({
            "index_dataframe": index,
            "colonne": col,
            "valeur_originale": valeur
        })

df_anomalies_conversion = pd.DataFrame(anomalies_conversion)

print(f"Nombre total de valeurs non convertibles : {len(df_anomalies_conversion)}")

df_anomalies_conversion.head(30)

Nombre total de valeurs non convertibles : 1224


,index_dataframe,colonne,valeur_originale
0,30821,duration_seconds,2`
1,39519,duration_seconds,8`
2,64975,duration_seconds,0.5`
3,48461,latitude,33q.200088
4,166,datetime,10/10/2005 24:00
5,316,datetime,10/11/1994 24:00
6,417,datetime,10/11/2006 24:00
7,487,datetime,10/11/2012 24:00
8,567,datetime,10/1/1972 24:00
9,607,datetime,10/1/1981 24:00


## XI. Enregistrer les anomalies de conversion

In [18]:
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
df_resume_conversion.to_csv(
    OUTPUT_DIR / "resume_conversions.csv",
    index=False
)
df_anomalies_conversion.to_csv(
    OUTPUT_DIR / "anomalies_conversion.csv",
    index=False
)

## XII.  Vérifier les anomalies numériques « métier »

In [21]:
anomalies_metier = pd.DataFrame({
    "duree_negative": df["duration_seconds"] < 0,
    "latitude_hors_plage": ~df["latitude"].between(-90, 90),
    "longitude_hors_plage": ~df["longitude"].between(-180, 180),
})

anomalies_metier.sum()

duree_negative          0
latitude_hors_plage     1
longitude_hors_plage    0
dtype: int64

## XIII. Afficher les lignes concernées

In [22]:
colonnes_a_afficher = [
    "datetime",
    "city",
    "state",
    "country",
    "duration_seconds",
    "latitude",
    "longitude",
]

df.loc[
    anomalies_metier.any(axis=1),
    colonnes_a_afficher
].head(30)

,datetime,city,state,country,duration_seconds,latitude,longitude
48461,1974-05-22 05:30:00,mescalero indian reservation,nm,,180.0,NaN,-105.624152


## XIV. Rechercher des dates incohérentes

In [23]:
print("Date minimale d'observation :", df["datetime"].min())
print("Date maximale d'observation :", df["datetime"].max())

print("Date minimale de publication :", df["date_posted"].min())
print("Date maximale de publication :", df["date_posted"].max())

Date minimale d'observation : 1906-11-11 00:00:00
Date maximale d'observation : 2014-05-08 18:45:00
Date minimale de publication : 1998-03-07 00:00:00
Date maximale de publication : 2014-05-08 00:00:00


In [24]:
dates_incoherentes = df[
    df["datetime"].notna()
    & df["date_posted"].notna()
    & (df["date_posted"] < df["datetime"])
]

print("Nombre de publications antérieures à l'observation :", len(dates_incoherentes))

dates_incoherentes[
    [
        "datetime",
        "date_posted",
        "city",
        "country",
        "comments"
    ]
].head(20)

Nombre de publications antérieures à l'observation : 407


,datetime,date_posted,city,country,comments
252,2011-10-10 02:00:00,2011-10-10,prescott valley,us,Craft boomerang shape.2:00am duration hours. ...
256,2011-10-10 15:00:00,2011-10-10,groton,us,Small shiny object seen in sky while driving o...
391,2005-10-11 03:00:00,2005-10-11,eagan,us,((NUFORC Note: Possible star. PD)) Bright c...
1135,2001-10-12 19:40:00,2001-10-12,sainte-suzanne (near switzerland) (france),,The craft was too dark&#44 but behind i have s...
1838,2013-10-14 00:35:00,2013-10-14,forked river,us,Red/gold fireball haze around it stayed same ...
1839,2013-10-14 01:40:00,2013-10-14,albuquerque,us,3 flashing green lights in formatiom of a tria...
1840,2013-10-14 11:47:00,2013-10-14,hampton,us,White cylinder with red light at back. Didn&#3...
2253,2002-10-15 20:30:00,2002-10-15,darwin (nt&#44 australia),au,two loud noises&#44 saw unusual lights in the ...
2258,2002-10-15 22:30:00,2002-10-15,slough&#44 berksire (uk/england),,There was just rotating white lights in the sk...
3323,2011-10-19 01:54:00,2011-10-19,denver,us,Silent V formation over Denver&#44 CO. ((NUFO...


## XV. Compter les valeurs manquantes après conversion

In [25]:
df[
    colonnes_numeriques + colonnes_dates
].isna().sum().sort_values(ascending=False)

datetime            1220
duration_seconds       5
latitude               1
longitude              0
date_posted            0
dtype: int64